# 📋 Notebook 1 — Ground Truth Generation
## TeleCopilot-AI Extraction Quality Experiment

This notebook creates **10 verified, gold-standard Ground Truth `.md` files** from the raw data sources in `data/raw/`.
Each `.md` file contains:
- **YAML Frontmatter**: document metadata, critical entities, section hierarchy
- **Section Metadata Delimiters**: page numbers, order, parent-child structure
- **Lossless Text**: 100% of the document content including tables, metadata paragraphs, and (for DOCX/PDF) Gemini Vision-transcribed diagrams

### Output
> 📁 `data/ground_truth/<stem>.md` — one file per raw source document

---

## ⚙️ Section 1 — Setup & Environment

In [ ]:
import sys, os
from pathlib import Path

# ── Path configuration ──────────────────────────────────────────────────────
NOTEBOOK_DIR   = Path(os.getcwd())           # experiments/
PROJECT_ROOT   = NOTEBOOK_DIR.parent         # TeleCopilot-AI/
SRC_DIR        = PROJECT_ROOT / 'src'
RAW_DATA_DIR   = PROJECT_ROOT / 'data' / 'raw'
GT_OUTPUT_DIR  = PROJECT_ROOT / 'data' / 'ground_truth'
GT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [str(PROJECT_ROOT), str(SRC_DIR), str(NOTEBOOK_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✅ Project root : {PROJECT_ROOT}')
print(f'✅ Raw data dir : {RAW_DATA_DIR}')
print(f'✅ GT output dir: {GT_OUTPUT_DIR}')

In [ ]:
# ── Load environment variables ───────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')
if GEMINI_API_KEY:
    print(f'✅ GEMINI_API_KEY loaded ({GEMINI_API_KEY[:8]}...)')
else:
    print('⚠️  GEMINI_API_KEY not set — PDF/DOCX vision OCR will use text-only fallback')

In [ ]:
# ── Initialise Gemini client ─────────────────────────────────────────────────
gemini_client = None
if GEMINI_API_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        gemini_client = genai.GenerativeModel('gemini-3.1-flash-lite-preview')
        print('✅ Gemini client ready (gemini-3.1-flash-lite-preview)')
    except Exception as e:
        print(f'⚠️  Gemini init failed: {e}')

# ── Import evaluation toolkit ────────────────────────────────────────────────
from extraction_eval.ground_truth_builder import GroundTruthBuilder, GroundTruthDocument
from extraction_eval.hitl import HITLReviewer

builder  = GroundTruthBuilder(gemini_client=gemini_client)
reviewer = HITLReviewer(report_dir=str(PROJECT_ROOT / 'experiments' / 'reports'))
print('✅ GroundTruthBuilder and HITLReviewer ready')

## 📁 Section 2 — Source Discovery

In [ ]:
import pandas as pd

SUPPORTED_EXTS = {'.html', '.htm', '.xml', '.docx', '.doc', '.pdf'}
raw_files = sorted(
    [f for f in RAW_DATA_DIR.iterdir() if f.suffix.lower() in SUPPORTED_EXTS]
)

catalogue = []
for f in raw_files:
    size_kb = f.stat().st_size / 1024
    catalogue.append({
        'file': f.name,
        'format': f.suffix.lower().lstrip('.'),
        'size_kb': round(size_kb, 1),
        'size_mb': round(size_kb / 1024, 2),
    })

df = pd.DataFrame(catalogue)
print(f'Found {len(df)} source files:\n')
display(df.style.set_caption('Raw Data Catalogue').format({'size_kb': '{:.1f}', 'size_mb': '{:.2f}'}))

## 🏗️ Section 3 — Ground Truth Generation (All Formats)

In [ ]:
from IPython.display import display, HTML
import traceback

ground_truths: dict[str, GroundTruthDocument] = {}
build_errors: dict[str, str] = {}

for raw_file in raw_files:
    print(f'\n🔧 Building GT for: {raw_file.name} ({raw_file.stat().st_size / 1024:.0f} KB)')
    try:
        gt = builder.build(str(raw_file))
        ground_truths[raw_file.name] = gt
        print(f'   ✅ {gt.title}')
        print(f'   📌 Sections: {len(gt.sections)} | Words: {gt.total_words:,} '
              f'| Tables cells: {gt.total_table_cells} | Media: {gt.total_media_count}')
    except Exception as e:
        build_errors[raw_file.name] = str(e)
        print(f'   ❌ ERROR: {e}')
        traceback.print_exc()

print(f'\n✅ Built {len(ground_truths)}/{len(raw_files)} ground truth documents')
if build_errors:
    print(f'⚠️  Errors in: {", ".join(build_errors.keys())}')

In [ ]:
# Preview first 500 chars of each Ground Truth
for fname, gt in ground_truths.items():
    md_preview = builder.to_markdown(gt)
    display(HTML(f"""
    <details style='border:1px solid #e2e8f0;border-radius:6px;margin:8px 0;padding:4px'>
      <summary style='padding:8px 12px;cursor:pointer;font-weight:bold;font-family:monospace'>
          📄 {fname} — {gt.file_format.upper()} | {len(gt.sections)} sections | {gt.total_words:,} words
      </summary>
      <pre style='padding:12px;background:#f8fafc;font-size:11px;overflow-x:auto;max-height:300px'>{md_preview[:800]}...</pre>
    </details>
    """))

## ✅ Section 4 — Ground Truth Self-Validation

Validates each generated Ground Truth against the raw file to detect any completeness issues **before** saving.
Each check either passes ✅ or flags ⚠️ for human review.

In [ ]:
import re
from extraction_eval.metrics import extract_critical_entities

validation_results = []

for fname, gt in ground_truths.items():
    raw_file = RAW_DATA_DIR / fname
    raw_size_bytes = raw_file.stat().st_size
    gt_char_count = len(gt.raw_text)
    md_content = builder.to_markdown(gt)

    issues = []
    checks = {}

    # ── Check 1: Character coverage (GT chars vs raw file size) ──────────────
    # For text formats: GT chars should be at least 30% of raw bytes
    # (HTML/XML have lots of markup overhead, PDFs have binary overhead)
    min_coverage_ratio = {'html': 0.10, 'xml': 0.08, 'docx': 0.05, 'pdf': 0.02}
    min_ratio = min_coverage_ratio.get(gt.file_format, 0.05)
    coverage = gt_char_count / raw_size_bytes
    checks['char_coverage'] = coverage >= min_ratio
    if not checks['char_coverage']:
        issues.append(f'Low character coverage: {coverage:.2%} (min {min_ratio:.0%})')

    # ── Check 2: Minimum section count ───────────────────────────────────────
    checks['min_sections'] = len(gt.sections) >= 1
    if not checks['min_sections']:
        issues.append('No sections extracted')

    # ── Check 3: Critical entity count matches frontmatter ───────────────────
    found_entities = extract_critical_entities(gt.raw_text)
    entity_count = sum(len(v) for v in found_entities.values())
    checks['has_entities'] = entity_count > 0 or gt.file_format not in ['xml']
    if not checks['has_entities']:
        issues.append('No critical entities (numbers, SLA codes, prices) found in GT text')

    # ── Check 4: YAML frontmatter is valid ────────────────────────────────────
    try:
        import yaml
        fm_match = re.match(r'^---\n(.+?)\n---', md_content, re.DOTALL)
        if fm_match:
            yaml.safe_load(fm_match.group(1))
            checks['valid_yaml'] = True
        else:
            checks['valid_yaml'] = False
            issues.append('YAML frontmatter not found or malformed')
    except Exception as e:
        checks['valid_yaml'] = False
        issues.append(f'YAML parse error: {e}')

    # ── Check 5: Section metadata delimiters present ──────────────────────────
    delimiter_count = len(re.findall(r'<!-- section:', md_content))
    checks['has_section_delimiters'] = delimiter_count == len(gt.sections)
    if not checks['has_section_delimiters']:
        issues.append(f'Section delimiter mismatch: {delimiter_count} delimiters vs {len(gt.sections)} sections')

    overall_pass = all(checks.values())
    validation_results.append({
        'file': fname,
        'format': gt.file_format,
        'sections': len(gt.sections),
        'words': gt.total_words,
        'entities': entity_count,
        'char_coverage': f'{coverage:.2%}',
        'status': '✅ PASS' if overall_pass else f'⚠️ ISSUES: {len(issues)}',
        'issues': issues,
    })

    status = '✅ PASS' if overall_pass else f'⚠️  {len(issues)} issue(s)'
    print(f'{status} — {fname}')
    for issue in issues:
        print(f'       → {issue}')

In [ ]:
# Display validation summary table
val_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'issues'} for r in validation_results
])
display(val_df.style.set_caption('Ground Truth Validation Results'))

# Flag documents needing HITL review
flagged = [r for r in validation_results if '⚠️' in r['status']]
if flagged:
    print(f'\n⚠️  {len(flagged)} documents need HITL review before saving:')
    for r in flagged:
        print(f'  • {r["file"]}: {r["issues"]}')
else:
    print('\n✅ All Ground Truth documents passed validation!')

## 👁️ Section 5 — Human-In-The-Loop (HITL) Sign-off

In [ ]:
# ── Interactive spot-check: select a document to review ─────────────────────
# Change this to review any file by name
REVIEW_FILE = raw_files[6].name   # default: first file

if REVIEW_FILE in ground_truths:
    gt = ground_truths[REVIEW_FILE]
    raw_path = RAW_DATA_DIR / REVIEW_FILE
    
    # Read raw file text for comparison
    raw_preview = ''
    try:
        with open(raw_path, 'r', encoding='utf-8', errors='ignore') as f:
            raw_preview = f.read(3000)
    except Exception:
        raw_preview = '[Binary file — cannot display raw]'

    reviewer.display_diff(
        extracted_text=gt.raw_text,
        ground_truth_text=raw_preview,
        title=f'Ground Truth Preview — {REVIEW_FILE}'
    )

    # If DOCX or PDF: display images if available
    if gt.embedded_images:
        reviewer.display_image_alongside_text(
            images=gt.embedded_images,
            extracted_text=gt.raw_text,
            title=f'Embedded Media — {REVIEW_FILE}'
        )

In [ ]:
# ── Record your HITL review for each document ────────────────────────────────
# Repeat this cell (or create copies) for any documents that need review.
# Actions: 'APPROVE', 'FLAG', 'OVERRIDE_SCORE'

for fname, gt in ground_truths.items():
    val_entry = next((r for r in validation_results if r['file'] == fname), {})
    action = 'FLAG' if '⚠️' in val_entry.get('status', '') else 'APPROVE'
    
    reviewer.record_review(
        file_name=fname,
        file_type=gt.file_format,
        action=action,
        comment='Auto-reviewed based on validation checks' if action == 'APPROVE' else 'Flagged for manual inspection',
        flagged_issues=val_entry.get('issues', []),
    )

## 💾 Section 6 — Export Ground Truth Files

In [ ]:
saved_files = []
skipped_files = []

for fname, gt in ground_truths.items():
    # Only save documents with no blocking issues, or that were manually approved
    val_entry = next((r for r in validation_results if r['file'] == fname), {})
    human_review = next(
        (r for r in reviewer.audit_log if r.file_name == fname), None
    )
    
    if human_review and human_review.action == 'FLAG':
        print(f'⏭️  Skipping flagged: {fname}')
        skipped_files.append(fname)
        continue
    
    out_path = builder.save(gt, str(GT_OUTPUT_DIR))
    saved_files.append(out_path)
    print(f'💾 Saved: {Path(out_path).name}')

print(f'\n✅ Saved {len(saved_files)} ground truth files to: {GT_OUTPUT_DIR}')
if skipped_files:
    print(f'⚠️  Skipped {len(skipped_files)} flagged files: {", ".join(skipped_files)}')

# Save HITL audit log
reviewer.save_audit_log('gt_generation_hitl_log.json')

In [ ]:
# ── Final verification: list all saved files with sizes ─────────────────────
print('\n📁 Ground Truth Files Created:')
print('─' * 60)
total_gt_size = 0
for p in sorted(GT_OUTPUT_DIR.glob('*.md')):
    size = p.stat().st_size
    total_gt_size += size
    print(f'  {p.name:<55} {size/1024:>6.1f} KB')
print('─' * 60)
print(f'  Total: {total_gt_size/1024:.1f} KB')
print('\n✅ Notebook 1 complete — proceed to 02_extraction_quality_evaluation.ipynb')